In [18]:
import pickle

from keras.src.applications.nasnet import NASNet
from scipy import stats
import numpy as np
import statsmodels.stats.api as sms
from utils import metrics
from statsmodels.stats.multitest import multipletests
import pingouin as pg

In [19]:
with open('all_metrics_single.pkl', 'rb') as f:
    all_metrics_single = pickle.load(f)

In [20]:
def is_data_normal(data, alpha=0.05):
    if len(data) > 5000:
        stat, p_value = stats.kstest(data, 'norm')
    else:
        stat, p_value = stats.shapiro(data)
    return p_value > alpha

In [36]:
def effect_size_interpreter(effect_size, effect_size_type):
    if effect_size_type == 'Cohen_d':
        if np.abs(effect_size) < 0.2:
            effect_interpret = 'negligible'
        elif 0.2 <= np.abs(effect_size) < 0.5:
            effect_interpret = 'small'
        elif 0.5 <= np.abs(effect_size) < 0.8:
            effect_interpret = 'medium'
        else:
            effect_interpret = 'large'
    elif effect_size_type == 'RBC':  # Rank-Biserial Correlation
        if np.abs(effect_size) < 0.1:
            effect_interpret = 'negligible'
        elif 0.1 <= np.abs(effect_size) < 0.3:
            effect_interpret = 'small'
        elif 0.3 <= np.abs(effect_size) < 0.5:
            effect_interpret = 'medium'
        else:
            effect_interpret = 'large'
    elif effect_size_type in ['Rosenthal']:
        if np.abs(effect_size) < 0.1:
            effect_interpret = 'negligible'
        elif 0.1 <= np.abs(effect_size) < 0.3:
            effect_interpret = 'small'
        elif 0.3 <= np.abs(effect_size) < 0.5:
            effect_interpret = 'medium'
        else:
            effect_interpret = 'large'
    else:
        effect_interpret = np.nan
    
    return effect_interpret
        

In [42]:
def compare_metrics_with_none_group(data, alpha=0.05, correction_method='bh'):
    baseline = data['none.pkl']
    other_conditions = {k: v for k, v in data.items() if k != 'none.pkl'}
    metrics_data = baseline.keys()
    results = {}
    unpaired_conditions = ['lower', 'upper', 'complex', 'sitting']

    for metric in metrics_data:
        results[metric] = []
        p_values = []
        test_results = []
        base_data = np.array(baseline[metric]).flatten()

        normal_base = is_data_normal(base_data)

        for condition_name, condition_data in other_conditions.items():
            aug_data = np.array(condition_data[metric]).flatten()

            normal_aug = is_data_normal(aug_data)

            if condition_name in unpaired_conditions:
                # Unpaired data
                base_mean = np.mean(base_data)
                aug_mean = np.mean(aug_data)
                diff_percent = ((base_mean - aug_mean) / base_mean) * 100 if base_mean != 0 else 0
                diff_total = aug_mean - base_mean

                if normal_base and normal_aug:
                    t_stat, p_value = stats.ttest_ind(aug_data, base_data, equal_var=False)
                    test_used = 'Independent Samples t-test'
                else:
                   # Mann-Whitney U test with Scipy
                    u_stat, p_value = stats.mannwhitneyu(aug_data, base_data, alternative='two-sided')
                    test_used = 'Mann-Whitney U'

                p_values.append(p_value)

                # Effect size and CI
                if test_used == 'Independent Samples t-test':
                    effsize_results = pg.compute_effsize(aug_data, base_data, paired=False, eftype='cohen', confidence=0.95)
                    effect_size = effsize_results['cohen']
                    ci_lower = effsize_results['CI95%'][0]
                    ci_upper = effsize_results['CI95%'][1]
                    effect_interpret = effect_size_interpreter(effect_size, 'Cohen_d')
                else:
                    # Calculate Rank-Biserial Correlation (RBC) 
                    n1 = len(aug_data)
                    n2 = len(base_data)
                    effect_size = (2 * u_stat) / (n1 * n2) - 1
                    ci_lower, ci_upper = None, None  # CI for rbc is not standard
                    effect_interpret = effect_size_interpreter(effect_size, 'RBC')

                # Bootstrap CI for mean difference
                conf_int = bootstrap_confidence_interval(base_data, aug_data, n_resamples=1000, alpha=alpha, paired=False)

                # Significant flag will be updated after correction
                test_results.append({
                    'augmentation': condition_name,
                    'test_used': test_used,
                    'Significant': None,
                    'p_value': p_value,
                    'confidence_interval': conf_int,
                    'effect_size': effect_size,
                    'effect_size_ci': (ci_lower, ci_upper),
                    'effect_interpretation': effect_interpret,  
                    'difference_percent': diff_percent,
                    'difference_total': diff_total,
                    # Paired metrics set to None
                    'bias': None,
                    'pearsons_r': None,
                    'icc': None,
                    'icclb': None,
                    'iccup': None,
                    'sem': None,
                    'mdc': None
                })
            else:
                # Paired data
                if len(base_data) != len(aug_data):
                    print(f"Warning: Length mismatch for {metric} - {condition_name}. Expected paired data but lengths differ.")
                    continue

                base_mean = np.mean(base_data)
                aug_mean = np.mean(aug_data)
                diff_percent = ((base_mean - aug_mean) / base_mean) * 100 if base_mean != 0 else 0
                diff_total = aug_mean - base_mean

                if normal_base and normal_aug and is_data_normal(base_data - aug_data):
                    t_stat, p_value = stats.ttest_rel(aug_data, base_data)
                    test_used = 'Paired t-test'
                else:
                    t_stat, p_value = stats.wilcoxon(aug_data, base_data)
                    test_used = 'Wilcoxon signed-rank test'

                p_values.append(p_value)

                # Effect size and CI
                if test_used == 'Paired t-test':
                    effsize_results = pg.compute_effsize(aug_data, base_data, paired=True, eftype='cohen', confidence=0.95)
                    effect_size = effsize_results['cohen']
                    ci_lower = effsize_results['CI95%'][0]
                    ci_upper = effsize_results['CI95%'][1]
                    effect_interpret = effect_size_interpreter(effect_size, 'Cohen_d')
                else:
                    # Calculate effect size for Wilcoxon signed-rank test (Rosenthal's r)
                    z_stat = stats.norm.ppf(1 - p_value / 2)
                    n = len(base_data)
                    effect_size = z_stat / np.sqrt(n)
                    (ci_lower, ci_upper) = None, None  
                    effect_interpret = effect_size_interpreter(effect_size, 'Rosenthal')

                # Bootstrap CI for mean difference
                conf_int = bootstrap_confidence_interval(base_data, aug_data, n_resamples=1000, alpha=alpha, paired=True)

                # Paired metrics calculations (assuming you have these functions)
                icc, icclb, iccup = metrics.calculate_icc(base_data, aug_data)
                sem, mdc = metrics.calculate_sem_mdc(np.std(aug_data - base_data), icc)
                bias = metrics.calculate_bias(base_data, aug_data)
                pearson_r = metrics.calculate_pearson_r(base_data, aug_data)

                # Significant flag will be updated after correction
                test_results.append({
                    'augmentation': condition_name,
                    'test_used': test_used,
                    'Significant': None,
                    'p_value': p_value,
                    'confidence_interval': conf_int,
                    'effect_size': effect_size,
                    'effect_size_ci': (ci_lower, ci_upper),
                    'effect_interpretation': effect_interpret,
                    'difference_percent': diff_percent,
                    'difference_total': diff_total,
                    'bias': bias,
                    'pearsons_r': pearson_r,
                    'icc': icc,
                    'icclb': icclb,
                    'iccup': iccup,
                    'sem': sem,
                    'mdc': mdc
                })

        # Apply multiple comparison correction
        bh_significant, corrected_p_values, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')

        # Update significance and corrected p-values
        for i, idx in enumerate(np.argsort(p_values)):
            test_results[idx]['Significant'] = bh_significant[i]
            test_results[idx]['corrected_p_value'] = corrected_p_values[i]
            # You can interpret effect size here if needed
            results[metric].append(test_results[idx])

    return results

In [43]:
def bootstrap_confidence_interval(data1, data2, n_resamples=1000, alpha=0.05, paired=True):
    """
    Calculate the bootstrap confidence interval for the mean difference.
    """
    boot_diffs = []
    np.random.seed(0)  # For reproducibility
    for _ in range(n_resamples):
        if paired:
            indices = np.random.choice(len(data1), len(data1), replace=True)
            resample1 = data1[indices]
            resample2 = data2[indices]
            diff = np.mean(resample1 - resample2)
        else:
            resample1 = np.random.choice(data1, size=len(data1), replace=True)
            resample2 = np.random.choice(data2, size=len(data2), replace=True)
            diff = np.mean(resample1) - np.mean(resample2)
        boot_diffs.append(diff)
    lower = np.percentile(boot_diffs, (alpha/2)*100)
    upper = np.percentile(boot_diffs, (1 - alpha/2)*100)
    return lower, upper

In [44]:
results = compare_metrics_with_none_group(all_metrics_single)